In [ ]:
import sqlite3
import tkinter as tk
from tkinter import messagebox, ttk

DB_NAME = "logistics.db"


def init_db():
    """Initializes the SQLite database with normalized tables, foreign keys, cascade deletes, and seed data."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    # Explicitly enable Foreign Key enforcement in SQLite
    cursor.execute("PRAGMA foreign_keys = ON;")

    # 1. ships table
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS ships (
            ship_id INTEGER PRIMARY KEY AUTOINCREMENT,
            ship_name TEXT UNIQUE NOT NULL,
            vessel_type TEXT NOT NULL
        )
    """
    )

    # 2. shipments table (FK -> ships with ON DELETE CASCADE)
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS shipments (
            shipment_id INTEGER PRIMARY KEY AUTOINCREMENT,
            ship_id INTEGER NOT NULL,
            container_count INTEGER NOT NULL,
            destination TEXT NOT NULL,
            FOREIGN KEY (ship_id) REFERENCES ships (ship_id) ON DELETE CASCADE
        )
    """
    )

    # 3. tracking_logs table (FK -> shipments with ON DELETE CASCADE)
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS tracking_logs (
            log_id INTEGER PRIMARY KEY AUTOINCREMENT,
            shipment_id INTEGER NOT NULL,
            status_update TEXT NOT NULL,
            weight_tons REAL NOT NULL,
            FOREIGN KEY (shipment_id) REFERENCES shipments (shipment_id) ON DELETE CASCADE
        )
    """
    )

    # Seed sample entries if tables are empty
    cursor.execute("SELECT COUNT(*) FROM ships")
    if cursor.fetchone()[0] == 0:
        cursor.execute(
            "INSERT INTO ships (ship_name, vessel_type) VALUES (?, ?)",
            ("Ocean Trader", "Container Ship"),
        )
        cursor.execute(
            "INSERT INTO ships (ship_name, vessel_type) VALUES (?, ?)",
            ("Pacific Voyager", "Cargo Vessel"),
        )

        cursor.execute(
            "INSERT INTO shipments (ship_id, container_count, destination) VALUES (?, ?, ?)",
            (1, 150, "Singapore Port"),
        )
        cursor.execute(
            "INSERT INTO shipments (ship_id, container_count, destination) VALUES (?, ?, ?)",
            (2, 200, "Rotterdam Port"),
        )

        cursor.execute(
            "INSERT INTO tracking_logs (shipment_id, status_update, weight_tons) VALUES (?, ?, ?)",
            (1, "In Transit - Indian Ocean", 2450.5),
        )
        cursor.execute(
            "INSERT INTO tracking_logs (shipment_id, status_update, weight_tons) VALUES (?, ?, ?)",
            (2, "Docked - Preparing Unload", 3100.0),
        )

        conn.commit()

    conn.close()


class LogisticsDashboard:

    def __init__(self, root):
        self.root = root
        self.root.title(
            "Global Logistics: Cargo Fleet & Shipping Manifest Dashboard"
        )
        self.root.geometry("860x620")

        # Initialize backend database
        init_db()

        # Input variables
        self.ship_name_var = tk.StringVar()
        self.destination_var = tk.StringVar()
        self.weight_var = tk.StringVar()

        self.create_widgets()
        self.refresh_all()

    def create_widgets(self):
        # Top Frame: Insert Panel & Delete Panel containers
        top_frame = ttk.Frame(self.root, padding=10)
        top_frame.pack(side=tk.TOP, fill=tk.X)

        # ----- 1. INSERT PANEL -----
        insert_lf = ttk.LabelFrame(
            top_frame, text=" Add New Manifest Entry ", padding=10
        )
        insert_lf.grid(row=0, column=0, padx=10, pady=5, sticky="nsew")

        ttk.Label(insert_lf, text="Cargo Ship Name:").grid(
            row=0, column=0, sticky="w", pady=3
        )
        ttk.Entry(insert_lf, textvariable=self.ship_name_var, width=22).grid(
            row=0, column=1, pady=3
        )

        ttk.Label(insert_lf, text="Destination:").grid(
            row=1, column=0, sticky="w", pady=3
        )
        ttk.Entry(insert_lf, textvariable=self.destination_var, width=22).grid(
            row=1, column=1, pady=3
        )

        ttk.Label(insert_lf, text="Cargo Weight (Tons):").grid(
            row=2, column=0, sticky="w", pady=3
        )
        ttk.Entry(insert_lf, textvariable=self.weight_var, width=22).grid(
            row=2, column=1, pady=3
        )

        ttk.Button(insert_lf, text="Add Record", command=self.add_record).grid(
            row=3, column=0, columnspan=2, pady=10
        )

        # ----- 2. DELETE PANEL -----
        delete_lf = ttk.LabelFrame(
            top_frame, text=" Delete / Remove Entry ", padding=10
        )
        delete_lf.grid(row=0, column=1, padx=10, pady=5, sticky="nsew")

        ttk.Label(delete_lf, text="Delete Target Type:").grid(
            row=0, column=0, sticky="w", pady=3
        )
        self.delete_type_cb = ttk.Combobox(
            delete_lf,
            values=["Cargo Ship (Decommission Fleet)", "Single Shipment ID"],
            state="readonly",
            width=28,
        )
        self.delete_type_cb.grid(row=0, column=1, pady=3)
        self.delete_type_cb.current(0)
        self.delete_type_cb.bind("<<ComboboxSelected>>", self.update_delete_dropdown)

        ttk.Label(delete_lf, text="Select Item to Delete:").grid(
            row=1, column=0, sticky="w", pady=3
        )
        self.delete_item_cb = ttk.Combobox(
            delete_lf, state="readonly", width=28
        )
        self.delete_item_cb.grid(row=1, column=1, pady=3)

        ttk.Button(
            delete_lf, text="Delete Selected", command=self.delete_record
        ).grid(row=3, column=0, columnspan=2, pady=10)

        # ----- 3. TABBED DISPLAY PANEL -----
        notebook_frame = ttk.Frame(self.root, padding=10)
        notebook_frame.pack(side=tk.BOTTOM, fill=tk.BOTH, expand=True)

        self.notebook = ttk.Notebook(notebook_frame)
        self.notebook.pack(fill=tk.BOTH, expand=True)

        # Tab 1: Master JOIN View
        self.tab1 = ttk.Frame(self.notebook)
        self.notebook.add(self.tab1, text="Master Manifest (JOIN)")
        self.tree_join = self.create_treeview(
            self.tab1,
            [
                "Ship Name",
                "Destination",
                "Containers",
                "Status Update",
                "Weight (Tons)",
            ],
        )

        # Tab 2: Standalone Ships Table
        self.tab2 = ttk.Frame(self.notebook)
        self.notebook.add(self.tab2, text="Ships Table")
        self.tree_ships = self.create_treeview(
            self.tab2, ["Ship ID", "Ship Name", "Vessel Type"]
        )

        # Tab 3: Standalone Shipments Table
        self.tab3 = ttk.Frame(self.notebook)
        self.notebook.add(self.tab3, text="Shipments Table")
        self.tree_shipments = self.create_treeview(
            self.tab3,
            ["Shipment ID", "Ship ID", "Container Count", "Destination"],
        )

        # Tab 4: Standalone Tracking Logs Table
        self.tab4 = ttk.Frame(self.notebook)
        self.notebook.add(self.tab4, text="Tracking Logs Table")
        self.tree_logs = self.create_treeview(
            self.tab4, ["Log ID", "Shipment ID", "Status Update", "Weight (Tons)"]
        )

    def create_treeview(self, parent, columns):
        """Helper to build consistent tabular grids with scrollbars."""
        tree = ttk.Treeview(parent, columns=columns, show="headings")
        for col in columns:
            tree.heading(col, text=col)
            tree.column(col, width=145, anchor="center")
        scrollbar = ttk.Scrollbar(parent, orient=tk.VERTICAL, command=tree.yview)
        tree.configure(yscrollcommand=scrollbar.set)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        return tree

    def add_record(self):
        """Validates inputs and adds records with automatic ship mapping."""
        ship_name = self.ship_name_var.get().strip()
        destination = self.destination_var.get().strip()
        weight_str = self.weight_var.get().strip()

        # Input Validation: Blank Check
        if not ship_name or not destination or not weight_str:
            messagebox.showerror(
                "Input Error", "All text fields are required and cannot be blank."
            )
            return

        # Input Validation: Numeric & Range Check for Weight
        try:
            weight = float(weight_str)
            if weight < 0:
                messagebox.showerror(
                    "Input Error", "Cargo weight cannot be negative."
                )
                return
        except ValueError:
            messagebox.showerror(
                "Input Error", "Cargo weight must be a valid numeric value."
            )
            return

        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute("PRAGMA foreign_keys = ON;")

        # Logic check: Use existing ship ID if ship name exists, otherwise create new profile
        cursor.execute(
            "SELECT ship_id FROM ships WHERE LOWER(ship_name) = LOWER(?)",
            (ship_name,),
        )
        result = cursor.fetchone()

        if result:
            ship_id = result[0]
        else:
            cursor.execute(
                "INSERT INTO ships (ship_name, vessel_type) VALUES (?, ?)",
                (ship_name, "Standard Fleet"),
            )
            ship_id = cursor.lastrowid

        # Insert shipment record
        cursor.execute(
            "INSERT INTO shipments (ship_id, container_count, destination) VALUES (?, ?, ?)",
            (ship_id, 100, destination),
        )
        shipment_id = cursor.lastrowid

        # Insert tracking log record
        cursor.execute(
            "INSERT INTO tracking_logs (shipment_id, status_update, weight_tons) VALUES (?, ?, ?)",
            (shipment_id, "In Transit", weight),
        )

        conn.commit()
        conn.close()

        # Clear variables and refresh UI instantly
        self.ship_name_var.set("")
        self.destination_var.set("")
        self.weight_var.set("")
        self.refresh_all()
        messagebox.showinfo("Success", "Record added successfully!")

    def delete_record(self):
        """Deletes selected records and triggers cascading deletions."""
        selected_item = self.delete_item_cb.get()
        if not selected_item:
            messagebox.showerror(
                "Delete Error", "Please select a valid item to delete."
            )
            return

        target_type = self.delete_type_cb.get()
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute("PRAGMA foreign_keys = ON;")

        if "Cargo Ship" in target_type:
            ship_id = selected_item.split(" - ")[0]
            cursor.execute("DELETE FROM ships WHERE ship_id = ?", (ship_id,))
        else:
            shipment_id = selected_item.split(" ID: ")[1].split(" ")[0]
            cursor.execute(
                "DELETE FROM shipments WHERE shipment_id = ?", (shipment_id,)
            )

        conn.commit()
        conn.close()

        self.refresh_all()
        messagebox.showinfo(
            "Success",
            "Record dropped successfully. Cascading deletions applied.",
        )

    def update_delete_dropdown(self, event=None):
        """Populates dynamic dropdown selections for deletion."""
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        target_type = self.delete_type_cb.get()

        options = []
        if "Cargo Ship" in target_type:
            cursor.execute("SELECT ship_id, ship_name FROM ships")
            options = [f"{row[0]} - {row[1]}" for row in cursor.fetchall()]
        else:
            cursor.execute("SELECT shipment_id, destination FROM shipments")
            options = [
                f"Shipment ID: {row[0]} ({row[1]})" for row in cursor.fetchall()
            ]

        conn.close()
        self.delete_item_cb["values"] = options
        if options:
            self.delete_item_cb.current(0)
        else:
            self.delete_item_cb.set("")

    def refresh_all(self):
        """Refreshes dropdown items and re-loads all tab grids simultaneously."""
        self.update_delete_dropdown()

        for tree in [
            self.tree_join,
            self.tree_ships,
            self.tree_shipments,
            self.tree_logs,
        ]:
            for row in tree.get_children():
                tree.delete(row)

        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()

        # Tab 1: JOIN View Data
        join_query = """
            SELECT s.ship_name, sh.destination, sh.container_count, t.status_update, t.weight_tons
            FROM ships s
            JOIN shipments sh ON s.ship_id = sh.ship_id
            JOIN tracking_logs t ON sh.shipment_id = t.shipment_id
        """
        for row in cursor.execute(join_query).fetchall():
            self.tree_join.insert("", tk.END, values=row)

        # Tab 2: Raw Ships Data
        for row in cursor.execute("SELECT * FROM ships").fetchall():
            self.tree_ships.insert("", tk.END, values=row)

        # Tab 3: Raw Shipments Data
        for row in cursor.execute("SELECT * FROM shipments").fetchall():
            self.tree_shipments.insert("", tk.END, values=row)

        # Tab 4: Raw Tracking Logs Data
        for row in cursor.execute("SELECT * FROM tracking_logs").fetchall():
            self.tree_logs.insert("", tk.END, values=row)

        conn.close()


# Execution 
if __name__ == "__main__":
    try:
        if root.winfo_exists():
            root.destroy()
    except NameError:
        pass

    root = tk.Tk()
    app = LogisticsDashboard(root)
    root.mainloop()